# 05 pycavi Scamp

This notebook run SExtractor and Scamp over the mosaic images resulting from WCS solution retrieves by solve-field in notebook 4.

The flow is as follows:
- For each image with a wcs mosaic will first run SExtractor to create a FITS_LDAC catalogue of the stars in the field.
- Once the catalogue is created, Scamp is called. It will first extract the headers of the mosaic to create a .ahead file which will additionaly include airmass and filter header. Scamp will be called over the FITS_LDAC catalogue creating a .head file with the improved WCS headers.

Tested on:

- Ubuntu Linux laptop, takes approximately 1 minute by mosaic.
- Scientific Linux server
- Macbook Pro laptop running OSX Ventura

Tested with SExtractor 2.25 and Scamp 2.0.4. Please consider that older versions of SExtractor are called with /usr/bin/sex, but the newest (the one used here) are called with /usr/bin/source-extractor. I remember that in Chapman SExtractor must be called in the old way, so you will have to modified the SExtractor.run method to call the program correctly. The differences also includes the SExtractor parameters. Some parameters from the original file (for the older version) rises troubles when included in the new version. I have not tested if the 2.25 configuration works well on older versions.
If needed, I also include the old parameter configuration.

All the functions that relates with SExtractor or Scamp, like the creation of the configuration files, and the execution of the programs are inside their respective class within the file astromatic.py

In [1]:
import os
import sys

import glob
import logging

from astropy.io import fits

sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production
from pycavi import pycavi_tools
from pycavi import pycavi_astromatic_tools

This will be executed when the package is imported.


/home/lceledon/Software/pycavi/Pycavi-main/Pycavi-main_LC/notebooks/../papi/papi_sextractor.py:432: SyntaxWarning: invalid escape sequence '\.'
  _version_match = re.search(b"[Vv]ersion ([0-9\.])+", versionline)


In [2]:
# set the directory folders
scamp_path = '/usr/bin/scamp'
config_dir = os.getcwd()+'/'+'../config/'

config = pycavi_tools.read_pycavi_config(config_dir)
project_dir = config['Directories']['project_dir']
reduced_science_dir = config['Directories']['reduced_science_dir']  
reduced_std_dir = config['Directories']['reduced_std_dir']
log_dir = config['Directories']['project_logs_dir']

log_file_swarp = log_dir + "/pymeffpp_scamp.log"
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_file_swarp),
                              logging.StreamHandler()])
logging.getLogger().setLevel(logging.INFO)
    

In [3]:
# Determine and display data partitions available to scamp
# to select all filters the list below should be empty
# filters_to_scamp = ["NB_659"]
filters_to_scamp = []
reject_observations = []
        
files_to_scamp_sci = pycavi_astromatic_tools.scamp_lists(reduced_science_dir, reject_observations, filters_to_scamp)
observation_tags_list_sci = list(set(files_to_scamp_sci.keys()))
observation_tags_list_sci.sort()

files_to_scamp_std = pycavi_astromatic_tools.scamp_lists(reduced_std_dir, reject_observations, filters_to_scamp)
observation_tags_list_std = list(set(files_to_scamp_std.keys()))
observation_tags_list_std.sort()

files_to_scamp = files_to_scamp_sci | files_to_scamp_std
observation_tags_list = observation_tags_list_sci + observation_tags_list_std

# We can reduce the selection by setting the list
# of indices to choose. Leave it empty to choose all.
#
idx_selected = []

for i in range(len(observation_tags_list)):

    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue

    print("index: {} -- partition tag: {} -- nfiles: {} ".format(i, observation_tags_list[i], len(files_to_scamp[observation_tags_list[i]])))

    targets_to_scamp = files_to_scamp[observation_tags_list[i]]

    counter = 0
    for target in targets_to_scamp:
        print("  ----------> ", os.path.basename(target))

2024-10-30 14:20:54,967:INFO:scamp_lists: index: 0 -- observation_tag: CP_Pup/NB_659
2024-10-30 14:20:54,967:INFO:scamp_lists: index: 1 -- observation_tag: CP_Pup/r_SDSS


index: 0 -- partition tag: CP_Pup/NB_659 -- nfiles: 10 
  ---------->  OMEGA.2017-03-13T02:09:23.471_pre_wcs.fits
  ---------->  OMEGA.2017-03-13T02:02:54.140_pre_wcs.fits
  ---------->  OMEGA.2017-03-13T02:32:06.454_pre_wcs.fits
  ---------->  OMEGA.2017-03-13T02:06:08.281_pre_wcs.fits
  ---------->  OMEGA.2017-03-13T02:28:51.454_pre_wcs.fits
  ---------->  OMEGA.2017-03-13T02:25:36.543_pre_wcs.fits
  ---------->  OMEGA.2017-03-13T02:12:38.452_pre_wcs.fits
  ---------->  OMEGA.2017-03-13T02:22:22.303_pre_wcs.fits
  ---------->  OMEGA.2017-03-13T02:19:07.303_pre_wcs.fits
  ---------->  OMEGA.2017-03-13T02:15:53.282_pre_wcs.fits
index: 1 -- partition tag: CP_Pup/r_SDSS -- nfiles: 10 
  ---------->  OMEGA.2017-03-13T02:38:02.286_pre_wcs.fits
  ---------->  OMEGA.2017-03-13T02:39:07.286_pre_wcs.fits
  ---------->  OMEGA.2017-03-13T02:36:57.286_pre_wcs.fits
  ---------->  OMEGA.2017-03-13T02:35:52.825_pre_wcs.fits
  ---------->  OMEGA.2017-03-13T02:40:12.356_pre_wcs.fits
  ---------->  OME

In [5]:
# Here we run scamp on all data partitions or on a selected set of data partitions
idx_selected = [1]


# We determine the total number of files to scamp
ntot_files = 0
for i in range(len(observation_tags_list)):

    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue
    ntot_files += len(files_to_scamp[observation_tags_list[i]])

# Scamp loop
redo_scamp = True
redo_sextractor = True
counter = 0    
for i in range(len(observation_tags_list)):

    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue

    print("index: {} -- partition tag: {} -- nfiles/ntot: {}/{} ".format(i, \
                                                    observation_tags_list[i], \
                                                    len(files_to_scamp[observation_tags_list[i]]), \
                                                    ntot_files))

    targets_to_scamp = files_to_scamp[observation_tags_list[i]]
    target_name = observation_tags_list[i].split("/")[0]
    filter_name = observation_tags_list[i].split("/")[1]
    logging.info("target: {}".format(target_name))
    logging.info("filter: {}".format(filter_name))

    for target in targets_to_scamp:
        counter += 1

        # we check if the head file exists. It it does we skip
        #
        print("  ----------> ", os.path.basename(target))
        head_file = target[:-5]+'.head'
        cat_file = target[:-5]+'.cat'
        wcs_image_path =  '/'.join(target.split("/")[:-1])
        print("Scamp head file: {}".format(head_file))
        
        if os.path.isfile(head_file) and not redo_scamp:
            print("Scamp head file exists: {}. Skipping".format(head_file))
            continue
        elif os.path.isfile(head_file) and redo_scamp:
            os.remove(head_file)
            if os.path.isfile(cat_file) and redo_sextractor:
                os.remove(cat_file)
        
        # creates the Scamp folder where the log files will be located
        scamp_dir_path = "/".join(target.split("/")[:-4])+'/'+config['Directories']['scamp_dir_name']
        print("scamp_dir_path: {}".format(scamp_dir_path))
    
        try:
            logging.info("Atempting to create the Scamp_dir folder")
            os.mkdir(scamp_dir_path)
            logging.info("Folder created")
        except:
            logging.info("Folder already exists")

        log_file_scamp_sextractor = scamp_dir_path + "/pymeffpp_scamp_sextractor.log"
    
        logging.basicConfig(level=logging.INFO,
                        format="%(asctime)s:%(levelname)s:%(message)s",
                        handlers=[
                                  logging.FileHandler(log_file_scamp_sextractor),
                                  logging.StreamHandler()])
        logging.getLogger().setLevel(logging.INFO)

        logging.info("--------------------------------------------------------------------------------------")
        logging.info("Preparing to Sextract and Scamp fits image : {:d}/{:d}".format(counter, ntot_files))
        logging.info("Fits image : {}".format(target.split("/")[-1]))
    
        wcs_image_path =  '/'.join(target.split("/")[:-1])


    
        # SOURCE-EXTRACTOR ON OUTPUT OF SOLVE-FIELD
        #
        # we create the folders needed by sextractor
        # now the folder are created, it is moment to run sextractor first
        # first move the folder where the data are. This changing of directories
        # is needed because sorce-extractor cannot handle very long paths in its
        # input file
        #
        #sextractor_dir_path = reduced_science_dir+target_name+'/'+config['Directories']['sex_dir_name']
        sextractor_dir_path = "/".join(target.split("/")[:-4])+'/'+config['Directories']['sex_dir_name']
        try:
            logging.info("Atempting to create the Sextractor_dir folder: {}".format(reduced_science_dir+target_name))
            os.mkdir(sextractor_dir_path)
            logging.info("Folder created")
        except:
            logging.info("Folder already exists")
        
        # This flag can safely left to False unless you decide otherwise.
        # It can save some time.
        #
        ready_to_sextract = pycavi_astromatic_tools.prepare_to_sextract(target, redo_sextractor)

        # we tune some parameters for sextractr
        # we select bright stars for the comparison with the astrometric catalogue
        special_pars = {}
        special_pars["SATUR_KEY"] = "SATLEVEL"   # FITS keyword giving saturation value
        special_pars["DETECT_MINAREA"] = 5       # minimum number of pixels above threshold
        special_pars["DETECT_MAXAREA"] = 0       # maximun number of pixels above threshold
        special_pars["DETECT_THRESH"] = 10.0      # <sigmas> or <threshold>,<ZP> in mag.arcsec-2
        special_pars["ANALYSIS_THRESH"] = 10.0    # <sigmas> or <threshold>,<ZP> in mag.arcsec-2
        special_pars["SEEING_FWHM"] = 1.2        # stellar FWHM in arcsec (we give here typical not so good value)
        special_pars["PHOT_APERTURES"] = 5       # MAG_APER aperture diameter(s) in pixels
        special_pars["MAG_ZEROPOINT"] = 25.0     # magnitude zero-point
        special_pars["GAIN"] =  1.0              # detector gain in e-/ADU - if DN in e/s then 1.0
        special_pars["MEMORY_OBJSTACK"] = 6000   # number of objects in stack [3000]
        special_pars["MEMORY_PIXSTACK"] = 600000 # number of pixels in stack [300000]
    
        if ready_to_sextract:
            pycavi_astromatic_tools.run_sextractor_on_target(target[:-5]+".fits", sextractor_dir_path, special_pars)
    
        # sextractor is done
        logging.info("SExtractor finished on image {}".format(target.split("/")[-1]))

        # SCAMP ON OUTPUT OF SOLVE WITH THE SOURCE_EXTRACTOR CATALOGUE
        #
        # At this point Sextractor is finished and we can start to run Scamp
        # check if the Scamp .head file exists, same procedure as for Sextractor
        #scamp_dir_path = reduced_science_dir+target_name+'/Scamp_dir/'
        try:
            logging.info("Atempting to create the Scamp_dir folder: {}".format(scamp_dir_path))
            os.mkdir(scamp_dir_path)
            logging.info("Folder created")
        except:
            logging.info("Folder already exists")
        scamp_head_file = target[:-5]+".head"
    
        # At this point Sextractor is finished and we can start to run Scamp
        # check if the Scamp .head file exists, same procedure as for Sextractor

        # This flag must be kept True because the script fails if it finds 
        # the file from a previous run.
        redo_scamp = True
        ready_to_scamp = pycavi_astromatic_tools.prepare_to_scamp(target, redo_scamp)

        # Parameters that might be useful to adjust. This set assumes that solvefield
        # worked correctly.
        special_pars = {}    
        special_pars["ASTREF_CATALOG"] = "GAIA-DR2"    # NONE,FILE,USNO-A2,USNO-B1,GSC-2.3,
                                                      # TYCHO-2,UCAC-4,URAT-1,NOMAD-1,PPMX,
                                                      # CMC-15,2MASS,DENIS-3,SDSS-R9,SDSS-R12,
                                                      # IGSL,GAIA-DR1,GAIA-DR2,GAIA-EDR3,
                                                      # PANSTARRS-1, or ALLWISE
        special_pars["ASTREF_BAND"] = "REDDEST"           # Photom. band for astr.ref.magnitudes
                                                          # or DEFAULT, BLUEST, or REDDEST
        special_pars["ASTREFMAG_LIMITS"] = [-99.0, 99.0]  # Select magnitude range in ASTREF_BAND
        special_pars["PIXSCALE_MAXERR"] = 1.2             # Max scale-factor uncertainty
        special_pars["POSITION_MAXERR"] = 0.05            # Max positional uncertainty (arcmin)

        # FSE 2024-08-17: with SAME_CRVAL we get many inaccuracies. So it is better to leave at FIX_FOCALPLANE -- investigate
        # the hope is that with SAME_CRVAL we could determine a default solution as a function of filter
        # which should work sort of OK
        #
        special_pars["MOSAIC_TYPE"] = "FIX_FOCALPLANE"    # UNCHANGED, SAME_CRVAL, SHARE_PROJAXIS,
                                                          # FIX_FOCALPLANE or LOOSE
        special_pars["CROSSID_RADIUS"] = 1.0              # Cross-id initial radius (arcsec)
        special_pars["SN_THRESHOLDS"] = [5.0, 40.0]       # S/N thresholds (in sigmas) for all and high S/N sample
        special_pars["FWHM_THRESHOLDS"] = [3.0, 10.0]     # FWHM thresholds (in pixels) for sources
        special_pars["POSANGLE_MAXERR"] = 2.5             # Max position-angle uncertainty (deg)

        # ensuring the plots are in pdf
        special_pars['CHECKPLOT_DEV'] = 'PDF'             # diagnostic plots in pdf format, for better compatibility

        if ready_to_scamp:
            pycavi_astromatic_tools.run_scamp_on_target(target, scamp_dir_path, special_pars, scamp_path)

    
logging.info('\n\n -------------- DONE ---------------\n\n')  

2024-10-30 14:31:59,970:INFO:target: CP_Pup
2024-10-30 14:31:59,971:INFO:filter: r_SDSS
2024-10-30 14:31:59,972:INFO:Atempting to create the Scamp_dir folder
2024-10-30 14:31:59,972:INFO:Folder already exists
2024-10-30 14:31:59,972:INFO:--------------------------------------------------------------------------------------
2024-10-30 14:31:59,972:INFO:Preparing to Sextract and Scamp fits image : 1/10
2024-10-30 14:31:59,973:INFO:Fits image : OMEGA.2017-03-13T02:38:02.286_pre_wcs.fits
2024-10-30 14:31:59,973:INFO:Atempting to create the Sextractor_dir folder: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup
2024-10-30 14:31:59,973:INFO:Folder already exists
2024-10-30 14:31:59,974:INFO:Running Sextractor on image OMEGA.2017-03-13T02:38:02.286_pre_wcs.fits
2024-10-30 14:31:59,975:INFO:Moving to target's directory: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:38:02.286_pre
2024-10-30 14:31:59,976:INFO:sextracto

index: 1 -- partition tag: CP_Pup/r_SDSS -- nfiles/ntot: 10/10 
  ---------->  OMEGA.2017-03-13T02:38:02.286_pre_wcs.fits
Scamp head file: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:38:02.286_pre/OMEGA.2017-03-13T02:38:02.286_pre_wcs.head
scamp_dir_path: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/Scamp_dir
sextractor_base_dir: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:38:02.286_pre


2024-10-30 14:32:12,890:INFO:SExtractor finished on image OMEGA.2017-03-13T02:38:02.286_pre_wcs.fits
2024-10-30 14:32:12,890:INFO:Atempting to create the Scamp_dir folder: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/Scamp_dir
2024-10-30 14:32:12,891:INFO:Folder already exists
2024-10-30 14:32:12,891:INFO:Running Scamp on image OMEGA.2017-03-13T02:38:02.286_pre_wcs.fits
2024-10-30 14:32:12,891:INFO:write_scamp_ahead_file: OMEGA.2017-03-13T02:38:02.286_pre_wcs.fits
 dir_name: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:38:02.286_pre
2024-10-30 14:32:12,980:INFO:Moving to target's directory: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:38:02.286_pre
2024-10-30 14:32:12,981:INFO:scamp.config[ASTREF_CATALOG] = GAIA-DR2
2024-10-30 14:32:12,981:INFO:scamp.config[ASTREF_BAND] = REDDEST
2024-10-30 14:32:12,982:INFO:scamp.config[ASTREFMAG_LIMITS] =

py-sextractor.cat --> OMEGA.2017-03-13T02:38:02.286_pre_wcs.cat
run_command: cmdfile: scamp.txt
run_command: logfile: scamp.log
run_command: str_cmd: /usr/bin/scamp -c scamp.conf  OMEGA.2017-03-13T02:38:02.286_pre_wcs.cat
run_command: cwd: /home/lceledon/Investigacion/OmegaCam/Work_dir/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:38:02.286_pre


run_command: cmdfile: scamp.txt
run_command: str_cmd writting to command file scamp.txt
run_command: str_cmd written to command file scamp.txt
run_command: Running command: /usr/bin/scamp -c scamp.conf  OMEGA.2017-03-13T02:38:02.286_pre_wcs.cat
run_command: log_path set
run_command: univ_lines set
run_command: running...
run_command: done.


2024-10-30 14:32:25,882:INFO:Scamp finished without problems. Check the diagnosis plots for more information.
2024-10-30 14:32:25,883:INFO:Atempting to create the Scamp_dir folder
2024-10-30 14:32:25,883:INFO:Folder already exists
2024-10-30 14:32:25,884:INFO:--------------------------------------------------------------------------------------
2024-10-30 14:32:25,884:INFO:Preparing to Sextract and Scamp fits image : 2/10
2024-10-30 14:32:25,884:INFO:Fits image : OMEGA.2017-03-13T02:39:07.286_pre_wcs.fits
2024-10-30 14:32:25,885:INFO:Atempting to create the Sextractor_dir folder: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup
2024-10-30 14:32:25,885:INFO:Folder already exists
2024-10-30 14:32:25,885:INFO:Running Sextractor on image OMEGA.2017-03-13T02:39:07.286_pre_wcs.fits
2024-10-30 14:32:25,886:INFO:Moving to target's directory: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:39:07.286_pre
2024-10-30 14:32

...

2024-10-30 14:35:59,936:INFO:Scamp finished without problems. Check the diagnosis plots for more information.
2024-10-30 14:35:59,937:INFO:Atempting to create the Scamp_dir folder
2024-10-30 14:35:59,937:INFO:Folder already exists
2024-10-30 14:35:59,938:INFO:--------------------------------------------------------------------------------------
2024-10-30 14:35:59,938:INFO:Preparing to Sextract and Scamp fits image : 10/10
2024-10-30 14:35:59,938:INFO:Fits image : OMEGA.2017-03-13T02:42:22.447_pre_wcs.fits
2024-10-30 14:35:59,939:INFO:Atempting to create the Sextractor_dir folder: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup
2024-10-30 14:35:59,939:INFO:Folder already exists
2024-10-30 14:35:59,940:INFO:Running Sextractor on image OMEGA.2017-03-13T02:42:22.447_pre_wcs.fits
2024-10-30 14:35:59,941:INFO:Moving to target's directory: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:42:22.447_pre
2024-10-30 14:3

  ---------->  OMEGA.2017-03-13T02:42:22.447_pre_wcs.fits
Scamp head file: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:42:22.447_pre/OMEGA.2017-03-13T02:42:22.447_pre_wcs.head
scamp_dir_path: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/Scamp_dir
sextractor_base_dir: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:42:22.447_pre


2024-10-30 14:36:12,329:INFO:SExtractor finished on image OMEGA.2017-03-13T02:42:22.447_pre_wcs.fits
2024-10-30 14:36:12,329:INFO:Atempting to create the Scamp_dir folder: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/Scamp_dir
2024-10-30 14:36:12,330:INFO:Folder already exists
2024-10-30 14:36:12,330:INFO:Running Scamp on image OMEGA.2017-03-13T02:42:22.447_pre_wcs.fits
2024-10-30 14:36:12,330:INFO:write_scamp_ahead_file: OMEGA.2017-03-13T02:42:22.447_pre_wcs.fits
 dir_name: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:42:22.447_pre
2024-10-30 14:36:12,423:INFO:Moving to target's directory: /home/lceledon/Investigacion/OmegaCam/Work_dir//science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:42:22.447_pre
2024-10-30 14:36:12,424:INFO:scamp.config[ASTREF_CATALOG] = GAIA-DR2
2024-10-30 14:36:12,424:INFO:scamp.config[ASTREF_BAND] = REDDEST
2024-10-30 14:36:12,425:INFO:scamp.config[ASTREFMAG_LIMITS] =

py-sextractor.cat --> OMEGA.2017-03-13T02:42:22.447_pre_wcs.cat
run_command: cmdfile: scamp.txt
run_command: logfile: scamp.log
run_command: str_cmd: /usr/bin/scamp -c scamp.conf  OMEGA.2017-03-13T02:42:22.447_pre_wcs.cat
run_command: cwd: /home/lceledon/Investigacion/OmegaCam/Work_dir/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:42:22.447_pre


run_command: cmdfile: scamp.txt
run_command: str_cmd writting to command file scamp.txt
run_command: str_cmd written to command file scamp.txt
run_command: Running command: /usr/bin/scamp -c scamp.conf  OMEGA.2017-03-13T02:42:22.447_pre_wcs.cat
run_command: log_path set
run_command: univ_lines set
run_command: running...
run_command: done.


2024-10-30 14:36:26,225:INFO:Scamp finished without problems. Check the diagnosis plots for more information.
2024-10-30 14:36:26,225:INFO:

 -------------- DONE ---------------


